In [13]:
!git clone https://github.com/erwinzhang7/SSU-Data-Dash.git /content/SSU-Validation
%cd /content/SSU-Validation

fatal: destination path '/content/SSU-Validation' already exists and is not an empty directory.
/content/SSU-Validation


In [14]:
%cd /content/SSU-Validation

!grep -A 5 "def make_model" models.py

/content/SSU-Validation
def make_model(track):
    """Return a Gradient Boosting model for each track."""
    if track == "list_aware":
        return FeatureModel(features.list_aware_features)
    return FeatureModel(features.off_market_features)



In [15]:
%cd /content/SSU-Validation
%pip install -r requirements.txt

/content/SSU-Validation
Ignoring numpy: markers 'python_version >= "3.14"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 38.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 106.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 62.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 144.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 130.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 31.7 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully unins

In [16]:
%cd /content/SSU-Validation
!python run.py validate

/content/SSU-Validation
2025-12: trained on 139,931 earlier sales, predicted 1,238 homes
2026-01: trained on 141,169 earlier sales, predicted 1,058 homes
2026-02: trained on 142,227 earlier sales, predicted 1,442 homes
2026-03: trained on 143,669 earlier sales, predicted 1,816 homes
2026-04: trained on 145,485 earlier sales, predicted 2,224 homes
2026-05: trained on 147,709 earlier sales, predicted 2,291 homes
Wrote outputs/validation_predictions.csv


In [17]:
!python run.py report

# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 19.23% (18.92 to 19.57) | 14.87% (14.50 to 15.23) | +1.02% |
| list_aware | 10,069 | 5.82% (5.72 to 5.91) | 4.61% (4.50 to 4.71) | +0.68% |

MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.
A median signed error above 0 means the model tends to predict too high.
The intervals treat homes as independent. Homes sold in the same month share the same
market swings, so the real uncertainty is somewhat wider.

![Error by price range](../figures/error_by_price_range.png)

Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex and figures/error_by_price_range.png


In [ ]:
import pandas as pd

df = pd.read_csv("outputs/validation_predictions.csv")

df["APE"] = (
    abs(df["predicted_price"] - df["sold_price"])
    / df["sold_price"] * 100
)

monthly = (
    df.groupby(["month", "track"])["APE"]
    .agg(
        MdAPE="median",
        MAPE="mean",
        Homes="count"
    )
    .round(2)
    .reset_index()
)

display(monthly)

In [19]:
%cd /content/SSU-Validation

import numpy as np
import pandas as pd
import data
import features

from sklearn.ensemble import HistGradientBoostingRegressor

print("Ready for Hyperparameter Tuning!")

/content/SSU-Validation
Ready for Hyperparameter Tuning!


In [20]:
!git clone https://github.com/erwinzhang7/SSU-Data-Dash.git /content/SSU-Validation
%cd /content/SSU-Validation

fatal: destination path '/content/SSU-Validation' already exists and is not an empty directory.
/content/SSU-Validation


In [21]:
import shutil
from pathlib import Path

source = Path("/content")
destination = Path("/content/SSU-Validation")

for filename in [
    "models.py",
    "features.py",
    "run.py",
    "requirements.txt"
]:
    shutil.copy2(
        source / filename,
        destination / filename
    )
    print(f"Replaced: {filename}")

print("All files replaced successfully!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/models.py'

In [ ]:
%cd /content/SSU-Validation

!grep -A 5 "def make_model" models.py

In [ ]:
import numpy as np
import pandas as pd
import data
import features

from sklearn.ensemble import HistGradientBoostingRegressor

print("Ready for Hyperparameter Tuning!")

In [ ]:
# Different L2 regularization strengths
l2_values = [0.0, 1.0, 10.0, 100.0]

# Months used for hyperparameter tuning
tuning_months = pd.period_range(
    "2025-09",
    "2025-11",
    freq="M"
)

print("L2 values:", l2_values)
print("Tuning months:", tuning_months)

In [ ]:
def walk_forward_evaluate(params, track, listings, months):

    all_errors = []
    monthly_results = []

    # Choose features for each track
    if track == "off_market":
        feature_function = features.off_market_features
    else:
        feature_function = features.list_aware_features

    for month in months:

        # 1. Training data: only earlier sales
        train = listings[
            listings["date_sold"] < month.start_time
        ]

        # 2. Validation data: current month
        validation = listings[
            listings["date_sold"].dt.to_period("M") == month
        ]

        # 3. Create Gradient Boosting model
        model = HistGradientBoostingRegressor(
            random_state=0,
            **params
        )

        # 4. Train model on log(sold_price)
        X_train = feature_function(train)
        y_train = np.log(train["sold_price"])

        model.fit(X_train, y_train)

        # 5. Hide answers before prediction
        hidden = data.hide_answers(validation, track)
        X_val = feature_function(hidden)

        # 6. Predict sold prices
        predictions = np.exp(model.predict(X_val))

        # 7. Calculate prediction errors
        actual = validation["sold_price"].to_numpy()
        ape = np.abs(predictions - actual) / actual * 100

        all_errors.extend(ape)

        monthly_results.append({
            "Month": str(month),
            "MdAPE": np.median(ape),
            "MAPE": np.mean(ape),
            "Homes": len(validation)
        })

    return {
        "MdAPE": np.median(all_errors),
        "MAPE": np.mean(all_errors),
        "Monthly": pd.DataFrame(monthly_results)
    }

print("Walk-forward evaluation function is ready!")

In [ ]:
# Load historical data
listings = data.load("listings")

tuning_results = []

for track in ["off_market", "list_aware"]:

    for l2 in l2_values:

        print(f"Testing {track}, L2 = {l2}", flush=True)

        result = walk_forward_evaluate(
            params={"l2_regularization": l2},
            track=track,
            listings=listings,
            months=tuning_months
        )

        tuning_results.append({
            "Track": track,
            "L2": l2,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

tuning_results = pd.DataFrame(tuning_results)

display(
    tuning_results.sort_values(
        ["Track", "MdAPE"]
    ).round(3)
)

In [ ]:
best_params = (
    tuning_results
    .sort_values("MdAPE")
    .groupby("Track")
    .head(1)
    .reset_index(drop=True)
)

display(best_params)

In [ ]:
final_months = pd.period_range(
    "2025-12",
    "2026-05",
    freq="M"
)

final_results = []

for track in ["off_market", "list_aware"]:

    selected_l2 = float(
        best_params.loc[
            best_params["Track"] == track, "L2"
        ].iloc[0]
    )

    result = walk_forward_evaluate(
        params={"l2_regularization": selected_l2},
        track=track,
        listings=listings,
        months=final_months
    )

    final_results.append({
        "Track": track,
        "L2": selected_l2,
        "MdAPE": result["MdAPE"],
        "MAPE": result["MAPE"]
    })

final_results = pd.DataFrame(final_results)

display(final_results.round(3))

In [ ]:
original_mdape = {
    "off_market": 14.87,
    "list_aware": 4.66
}

final_results["Original MdAPE"] = (
    final_results["Track"].map(original_mdape)
)

final_results["Improvement (pp)"] = (
    final_results["Original MdAPE"]
    - final_results["MdAPE"]
)

display(final_results.round(3))

In [ ]:
from itertools import product

# Parameters for the second experiment
leaf_values = [7, 15, 31]
learning_rates = [0.05, 0.1]

# Keep other parameters at their original defaults
param_combinations = [
    {
        "max_leaf_nodes": leaf,
        "learning_rate": lr,
        "l2_regularization": 0.0
    }
    for leaf, lr in product(leaf_values, learning_rates)
]

display(pd.DataFrame(param_combinations))

In [ ]:
round2_results = []

for track in ["off_market", "list_aware"]:

    for i, params in enumerate(param_combinations):

        print(
            f"Testing {track}, configuration {i+1}/6",
            flush=True
        )

        result = walk_forward_evaluate(
            params=params,
            track=track,
            listings=listings,
            months=tuning_months
        )

        round2_results.append({
            "Track": track,
            **params,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

round2_results = pd.DataFrame(round2_results)

display(
    round2_results
    .sort_values(["Track", "MdAPE"])
    .round(3)
)

In [ ]:
round1 = tuning_results.rename(
    columns={"L2": "l2_regularization"}
).copy()

round1["max_leaf_nodes"] = 31
round1["learning_rate"] = 0.1
round1["Experiment"] = "L2 Regularization"

round2 = round2_results.copy()
round2["Experiment"] = "Tree Complexity"

all_results = pd.concat(
    [round1, round2],
    ignore_index=True
)

best_overall = (
    all_results
    .sort_values("MdAPE")
    .groupby("Track")
    .head(1)
    .reset_index(drop=True)
)

display(best_overall.round(3))

In [ ]:
round3_configs = [
    {"learning_rate": 0.10, "max_iter": 100, "max_leaf_nodes": 31},
    {"learning_rate": 0.05, "max_iter": 200, "max_leaf_nodes": 31},
    {"learning_rate": 0.05, "max_iter": 300, "max_leaf_nodes": 31},
    {"learning_rate": 0.10, "max_iter": 200, "max_leaf_nodes": 31},
]

display(pd.DataFrame(round3_configs))

In [ ]:
import os
import pandas as pd
import numpy as np

print("Project exists:",
      os.path.exists("/content/SSU-Validation"))

In [ ]:
!git clone https://github.com/erwinzhang7/SSU-Data-Dash.git /content/SSU-Validation
%cd /content/SSU-Validation

In [ ]:
from pathlib import Path

folder = Path("/content/SSU-Validation")

print("Folder exists:", folder.exists())
print("models.py:", (folder / "models.py").exists())
print("features.py:", (folder / "features.py").exists())
print("data.py:", (folder / "data.py").exists())
print("run.py:", (folder / "run.py").exists())

In [ ]:
%cd /content/SSU-Validation
!grep -A 5 "def make_model" models.py

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()

destination = Path("/content/SSU-Validation")

for filename in uploaded:
    shutil.copy2(filename, destination / filename)
    print("Restored:", filename)

In [ ]:
from pathlib import Path

folder = Path("/content/SSU-Validation")

files_to_replace = {
    "features (1).py": "features.py",
    "models (1).py": "models.py",
    "requirements (1).txt": "requirements.txt",
    "run (1).py": "run.py"
}

for uploaded_name, original_name in files_to_replace.items():
    source = folder / uploaded_name
    destination = folder / original_name

    if source.exists():
        source.replace(destination)
        print("Replaced:", original_name)
    else:
        print("Not found:", uploaded_name)

In [ ]:
%cd /content/SSU-Validation

!grep -A 5 "def make_model" models.py

In [1]:
import numpy as np
import pandas as pd
import data
import features

from sklearn.ensemble import HistGradientBoostingRegressor

listings = data.load("listings")

tuning_months = pd.period_range(
    "2025-09",
    "2025-11",
    freq="M"
)

print("Environment restored!")

ModuleNotFoundError: No module named 'data'

In [1]:
from pathlib import Path

folder = Path("/content/SSU-Validation")

print("Folder exists:", folder.exists())
print("data.py exists:", (folder / "data.py").exists())
print("models.py exists:", (folder / "models.py").exists())

Folder exists: False
data.py exists: False
models.py exists: False


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!git clone https://github.com/erwinzhang7/SSU-Data-Dash.git /content/SSU-Validation

%cd /content/SSU-Validation

Cloning into '/content/SSU-Validation'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 22 (delta 0), reused 0 (delta 0), pack-reused 19 (from 1)
Receiving objects: 100% (22/22), 22.60 KiB | 2.26 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/SSU-Validation


In [4]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

folder = Path("/content/SSU-Validation")

for name, content in uploaded.items():
    if name in [
        "models.py",
        "features.py",
        "run.py",
        "requirements.txt"
    ]:
        (folder / name).write_bytes(content)
        print("Restored:", name)

Saving features.py to features (1).py
Saving models.py to models (1).py
Saving requirements.txt to requirements (1).txt
Saving run.py to run (1).py


In [5]:
%cd /content/SSU-Validation

!grep -A 5 "def make_model" models.py

/content/SSU-Validation
def make_model(track):
    """Return a fresh, untrained model for "off_market" or "list_aware"."""
    if track == "list_aware":
        return ListPriceBaseline()
        # return FeatureModel(features.list_aware_features)   # try this instead
    return NearestSalesBaseline(k=10)


In [6]:
import shutil
from pathlib import Path

source = Path("/content/SSU-Validation")
backup = Path("/content/drive/MyDrive/SSU-Validation")

shutil.copytree(
    source,
    backup,
    dirs_exist_ok=True
)

print("Project permanently saved to Google Drive!")

Project permanently saved to Google Drive!


In [7]:
from pathlib import Path

model_file = Path(
    "/content/drive/MyDrive/SSU-Validation/models.py"
)

print(model_file.exists())

if model_file.exists():
    content = model_file.read_text()
    start = content.find("def make_model")
    print(content[start:start+400])

True
def make_model(track):
    """Return a fresh, untrained model for "off_market" or "list_aware"."""
    if track == "list_aware":
        return ListPriceBaseline()
        # return FeatureModel(features.list_aware_features)   # try this instead
    return NearestSalesBaseline(k=10)
    # return FeatureModel(features.off_market_features)       # try this instead


class ListPriceBaseline:
    """Li


In [8]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

project = Path("/content/SSU-Validation")
backup = Path("/content/drive/MyDrive/SSU-Validation")

for name, content in uploaded.items():
    if name in ["models.py", "features.py", "run.py", "requirements.txt"]:
        (project / name).write_bytes(content)
        (backup / name).write_bytes(content)
        print("Updated both copies:", name)

print("All files updated!")

Saving features.py to features (2).py
Saving models.py to models (2).py
Saving requirements.txt to requirements (2).txt
Saving run.py to run (2).py
All files updated!


In [9]:
from pathlib import Path

for folder in [
    Path("/content/SSU-Validation"),
    Path("/content/drive/MyDrive/SSU-Validation")
]:
    print("\nChecking:", folder)

    content = (folder / "models.py").read_text()
    start = content.find("def make_model")
    print(content[start:start+250])


Checking: /content/SSU-Validation
def make_model(track):
    """Return a fresh, untrained model for "off_market" or "list_aware"."""
    if track == "list_aware":
        return ListPriceBaseline()
        # return FeatureModel(features.list_aware_features)   # try this instead
    r

Checking: /content/drive/MyDrive/SSU-Validation
def make_model(track):
    """Return a fresh, untrained model for "off_market" or "list_aware"."""
    if track == "list_aware":
        return ListPriceBaseline()
        # return FeatureModel(features.list_aware_features)   # try this instead
    r


In [10]:
from pathlib import Path

project = Path("/content/SSU-Validation")
backup = Path("/content/drive/MyDrive/SSU-Validation")

old_models = project / "models.py"
content = old_models.read_text()

start = content.index("def make_model(track):")
end = content.index("\nclass ListPriceBaseline:", start)

new_function = '''def make_model(track):
    """Return a Gradient Boosting model for each track."""
    if track == "list_aware":
        return FeatureModel(features.list_aware_features)
    return FeatureModel(features.off_market_features)


'''

updated = content[:start] + new_function + content[end:]

(project / "models.py").write_text(updated)
(backup / "models.py").write_text(updated)

print("Both models.py files updated!")

Both models.py files updated!


In [11]:
for folder in [project, backup]:
    print("\nChecking:", folder)
    content = (folder / "models.py").read_text()
    start = content.index("def make_model")
    print(content[start:start+250])


Checking: /content/SSU-Validation
def make_model(track):
    """Return a Gradient Boosting model for each track."""
    if track == "list_aware":
        return FeatureModel(features.list_aware_features)
    return FeatureModel(features.off_market_features)



class ListPriceBaseline

Checking: /content/drive/MyDrive/SSU-Validation
def make_model(track):
    """Return a Gradient Boosting model for each track."""
    if track == "list_aware":
        return FeatureModel(features.list_aware_features)
    return FeatureModel(features.off_market_features)



class ListPriceBaseline


In [12]:
%cd /content/SSU-Validation

import numpy as np
import pandas as pd
import data
import features

from sklearn.ensemble import HistGradientBoostingRegressor

listings = data.load("listings")

tuning_months = pd.period_range(
    "2025-09", "2025-11", freq="M"
)

print("Environment restored!")

/content/SSU-Validation


listings.parquet: reconstructing file:   0%|          |  0.00B / 32.9MB            

listings.parquet: downloading bytes:           |  0.00B            

Environment restored!


In [13]:
def walk_forward_evaluate(params, track, listings, months):
    all_errors = []
    monthly_results = []

    if track == "off_market":
        feature_function = features.off_market_features
    else:
        feature_function = features.list_aware_features

    for month in months:
        train = listings[
            listings["date_sold"] < month.start_time
        ]
        validation = listings[
            listings["date_sold"].dt.to_period("M") == month
        ]

        model = HistGradientBoostingRegressor(
            random_state=0,
            **params
        )

        X_train = feature_function(train)
        y_train = np.log(train["sold_price"])
        model.fit(X_train, y_train)

        hidden = data.hide_answers(validation, track)
        X_val = feature_function(hidden)

        predictions = np.exp(model.predict(X_val))
        actual = validation["sold_price"].to_numpy()
        ape = np.abs(predictions - actual) / actual * 100

        all_errors.extend(ape)

        monthly_results.append({
            "Month": str(month),
            "MdAPE": np.median(ape),
            "MAPE": np.mean(ape),
            "Homes": len(validation)
        })

    return {
        "MdAPE": np.median(all_errors),
        "MAPE": np.mean(all_errors),
        "Monthly": pd.DataFrame(monthly_results)
    }

print("Walk-forward function restored!")

Walk-forward function restored!


In [14]:
round3_configs = [
    {"learning_rate": 0.10, "max_iter": 100, "max_leaf_nodes": 31},
    {"learning_rate": 0.05, "max_iter": 200, "max_leaf_nodes": 31},
    {"learning_rate": 0.05, "max_iter": 300, "max_leaf_nodes": 31},
    {"learning_rate": 0.10, "max_iter": 200, "max_leaf_nodes": 31},
]

display(pd.DataFrame(round3_configs))

,learning_rate,max_iter,max_leaf_nodes
0,0.10,100,31
1,0.05,200,31
2,0.05,300,31
3,0.10,200,31


In [15]:
round3_results = []

for track in ["off_market", "list_aware"]:
    for i, params in enumerate(round3_configs):

        print(
            f"Testing {track}, configuration {i+1}/4",
            flush=True
        )

        result = walk_forward_evaluate(
            params=params,
            track=track,
            listings=listings,
            months=tuning_months
        )

        round3_results.append({
            "Track": track,
            **params,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

round3_results = pd.DataFrame(round3_results)

display(
    round3_results
    .sort_values(["Track", "MdAPE"])
    .round(3)
)

Testing off_market, configuration 1/4
Testing off_market, configuration 2/4
Testing off_market, configuration 3/4
Testing off_market, configuration 4/4
Testing list_aware, configuration 1/4
Testing list_aware, configuration 2/4
Testing list_aware, configuration 3/4
Testing list_aware, configuration 4/4


,Track,learning_rate,max_iter,max_leaf_nodes,MdAPE,MAPE
7,list_aware,0.10,200,31,4.539,5.653
6,list_aware,0.05,300,31,4.546,5.671
5,list_aware,0.05,200,31,4.606,5.730
4,list_aware,0.10,100,31,4.633,5.739
3,off_market,0.10,200,31,14.192,18.562
2,off_market,0.05,300,31,14.696,19.137
0,off_market,0.10,100,31,15.274,19.730
1,off_market,0.05,200,31,15.290,19.786


In [16]:
from pathlib import Path

save_folder = Path(
    "/content/drive/MyDrive/SSU-Validation/reports"
)
save_folder.mkdir(parents=True, exist_ok=True)

round3_results.to_csv(
    save_folder / "member3_round3_results.csv",
    index=False
)

print("Round 3 results saved to Google Drive!")

Round 3 results saved to Google Drive!


In [17]:
final_months = pd.period_range(
    "2025-12",
    "2026-05",
    freq="M"
)

round3_final = []

params = {
    "learning_rate": 0.10,
    "max_iter": 200,
    "max_leaf_nodes": 31,
    "l2_regularization": 0.0
}

for track in ["off_market", "list_aware"]:

    print("Validating:", track, flush=True)

    result = walk_forward_evaluate(
        params=params,
        track=track,
        listings=listings,
        months=final_months
    )

    round3_final.append({
        "Track": track,
        **params,
        "MdAPE": result["MdAPE"],
        "MAPE": result["MAPE"]
    })

round3_final = pd.DataFrame(round3_final)

display(round3_final.round(3))

Validating: off_market
Validating: list_aware


,Track,learning_rate,max_iter,max_leaf_nodes,l2_regularization,MdAPE,MAPE
0,off_market,0.1,200,31,0.0,14.040,18.175
1,list_aware,0.1,200,31,0.0,4.618,5.782


In [18]:
round3_final.to_csv(
    "/content/drive/MyDrive/SSU-Validation/reports/member3_round3_final.csv",
    index=False
)

print("Final validation saved!")

Final validation saved!


In [19]:
HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_iter=200,
    max_leaf_nodes=31,
    l2_regularization=0.0,
    random_state=0
)

HistGradientBoostingRegressor(max_iter=200, random_state=0)

In [20]:
import pandas as pd
from pathlib import Path

print("Data ready:", "listings" in globals())
print("Function ready:", "walk_forward_evaluate" in globals())
print("Drive ready:", Path("/content/drive/MyDrive").exists())

Data ready: True
Function ready: True
Drive ready: True


In [21]:
from itertools import product

max_iter_values = [200, 300, 400]
l2_values_round4 = [0.0, 1.0, 10.0]

round4_configs = [
    {
        "learning_rate": 0.1,
        "max_iter": n,
        "max_leaf_nodes": 31,
        "l2_regularization": l2
    }
    for n, l2 in product(max_iter_values, l2_values_round4)
]

display(pd.DataFrame(round4_configs))

,learning_rate,max_iter,max_leaf_nodes,l2_regularization
0,0.1,200,31,0.0
1,0.1,200,31,1.0
2,0.1,200,31,10.0
3,0.1,300,31,0.0
4,0.1,300,31,1.0
5,0.1,300,31,10.0
6,0.1,400,31,0.0
7,0.1,400,31,1.0
8,0.1,400,31,10.0


In [22]:
import pandas as pd
from pathlib import Path

save_path = Path(
    "/content/drive/MyDrive/SSU-Validation/reports/member3_round4.csv"
)

save_path.parent.mkdir(parents=True, exist_ok=True)

# Restore previous progress if the experiment was interrupted
if save_path.exists():
    round4_results = pd.read_csv(save_path).to_dict("records")
else:
    round4_results = []

for track in ["off_market", "list_aware"]:
    for params in round4_configs:

        # Skip configurations already completed
        completed = any(
            r["Track"] == track
            and int(r["max_iter"]) == params["max_iter"]
            and float(r["l2_regularization"]) == params["l2_regularization"]
            for r in round4_results
        )

        if completed:
            continue

        print(
            f"Testing {track}, "
            f"max_iter={params['max_iter']}, "
            f"L2={params['l2_regularization']}",
            flush=True
        )

        result = walk_forward_evaluate(
            params=params,
            track=track,
            listings=listings,
            months=tuning_months
        )

        round4_results.append({
            "Track": track,
            **params,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

        # Save after every completed configuration
        pd.DataFrame(round4_results).to_csv(
            save_path,
            index=False
        )

round4_results = pd.DataFrame(round4_results)

display(
    round4_results
    .sort_values(["Track", "MdAPE"])
    .round(3)
)

print("Results saved to Google Drive!")

Testing off_market, max_iter=200, L2=0.0
Testing off_market, max_iter=200, L2=1.0
Testing off_market, max_iter=200, L2=10.0
Testing off_market, max_iter=300, L2=0.0
Testing off_market, max_iter=300, L2=1.0
Testing off_market, max_iter=300, L2=10.0
Testing off_market, max_iter=400, L2=0.0
Testing off_market, max_iter=400, L2=1.0
Testing off_market, max_iter=400, L2=10.0
Testing list_aware, max_iter=200, L2=0.0
Testing list_aware, max_iter=200, L2=1.0
Testing list_aware, max_iter=200, L2=10.0
Testing list_aware, max_iter=300, L2=0.0
Testing list_aware, max_iter=300, L2=1.0
Testing list_aware, max_iter=300, L2=10.0
Testing list_aware, max_iter=400, L2=0.0
Testing list_aware, max_iter=400, L2=1.0
Testing list_aware, max_iter=400, L2=10.0


,Track,learning_rate,max_iter,max_leaf_nodes,l2_regularization,MdAPE,MAPE
16,list_aware,0.1,400,31,1.0,4.451,5.604
13,list_aware,0.1,300,31,1.0,4.460,5.608
17,list_aware,0.1,400,31,10.0,4.464,5.587
10,list_aware,0.1,200,31,1.0,4.477,5.635
12,list_aware,0.1,300,31,0.0,4.482,5.624
15,list_aware,0.1,400,31,0.0,4.500,5.626
14,list_aware,0.1,300,31,10.0,4.502,5.608
11,list_aware,0.1,200,31,10.0,4.517,5.640
9,list_aware,0.1,200,31,0.0,4.539,5.653
6,off_market,0.1,400,31,0.0,13.242,17.575


Results saved to Google Drive!


In [23]:
final_months = pd.period_range(
    "2025-12", "2026-05", freq="M"
)

best_round4_configs = {
    "off_market": {
        "learning_rate": 0.1,
        "max_iter": 400,
        "max_leaf_nodes": 31,
        "l2_regularization": 0.0
    },
    "list_aware": {
        "learning_rate": 0.1,
        "max_iter": 400,
        "max_leaf_nodes": 31,
        "l2_regularization": 1.0
    }
}

round4_final = []

for track, params in best_round4_configs.items():

    print("Validating:", track, flush=True)

    result = walk_forward_evaluate(
        params=params,
        track=track,
        listings=listings,
        months=final_months
    )

    round4_final.append({
        "Track": track,
        **params,
        "MdAPE": result["MdAPE"],
        "MAPE": result["MAPE"]
    })

round4_final = pd.DataFrame(round4_final)

display(round4_final.round(3))

Validating: off_market
Validating: list_aware


,Track,learning_rate,max_iter,max_leaf_nodes,l2_regularization,MdAPE,MAPE
0,off_market,0.1,400,31,0.0,13.187,17.334
1,list_aware,0.1,400,31,1.0,4.574,5.748


In [24]:
HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_iter=400,
    max_leaf_nodes=31,
    l2_regularization=0,
    random_state=0
)

HistGradientBoostingRegressor(l2_regularization=0, max_iter=400, random_state=0)

In [25]:
HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_iter=400,
    max_leaf_nodes=31,
    l2_regularization=1,
    random_state=0
)

HistGradientBoostingRegressor(l2_regularization=1, max_iter=400, random_state=0)

In [26]:
monthly_round4 = []

for track, params in best_round4_configs.items():

    result = walk_forward_evaluate(
        params=params,
        track=track,
        listings=listings,
        months=final_months
    )

    monthly = result["Monthly"].copy()
    monthly["Track"] = track

    monthly_round4.append(monthly)

monthly_round4 = pd.concat(
    monthly_round4,
    ignore_index=True
)

display(monthly_round4.round(3))

,Month,MdAPE,MAPE,Homes,Track
0,2025-12,13.949,18.054,1238,off_market
1,2026-01,13.274,17.043,1058,off_market
2,2026-02,12.730,16.496,1442,off_market
3,2026-03,12.789,17.109,1816,off_market
4,2026-04,13.348,17.400,2224,off_market
5,2026-05,13.201,17.720,2291,off_market
6,2025-12,4.281,5.432,1238,list_aware
7,2026-01,4.179,5.301,1058,list_aware
8,2026-02,4.517,5.660,1442,list_aware
9,2026-03,4.468,5.695,1816,list_aware


In [27]:
import inspect
import data

print(inspect.getsource(data.hide_answers))

def hide_answers(homes, track):
    """Keep only the columns a test file of this track has.

    Validation uses this so that a model is never shown the answer (sold_price) or
    anything else that is only known after the sale. Off-market homes also lose
    their list price.
    """
    hidden = AFTER_SALE_COLUMNS + (["list_price"] if track == "off_market" else [])
    return homes.drop(columns=[c for c in hidden if c in homes.columns])



In [30]:
# Step 1: Check columns hidden after a sale
print("AFTER_SALE_COLUMNS:")
print(data.AFTER_SALE_COLUMNS)

# Step 2: Take 5 historical homes as an example
sample = listings.head(5)

# Step 3: Test both tracks
for track in ["off_market", "list_aware"]:

    hidden_sample = data.hide_answers(sample, track)

    print("\nTrack:", track)

    print(
        "sold_price visible:",
        "sold_price" in hidden_sample.columns
    )

    print(
        "list_price visible:",
        "list_price" in hidden_sample.columns
    )

AFTER_SALE_COLUMNS:
['sold_price', 'date_sold', 'days_on_market', 'external_avm_estimate', 'external_avm_estimate_date']

Track: off_market
sold_price visible: False
list_price visible: False

Track: list_aware
sold_price visible: False
list_price visible: True


In [31]:
import inspect
import features

print("=== OFF-MARKET FEATURES ===")
print(inspect.getsource(features.off_market_features))

print("\n=== LIST-AWARE FEATURES ===")
print(inspect.getsource(features.list_aware_features))

=== OFF-MARKET FEATURES ===
def off_market_features(homes):
    """Features for homes with no list price."""
    X = pd.DataFrame(index=homes.index)
    X["square_feet"] = square_feet(homes["house_area"])
    X["bedrooms"] = homes["bedrooms"]
    X["washrooms"] = homes["washrooms"]
    X["parking"] = homes["parking_total"]
    X["latitude"] = homes["latitude"]
    X["longitude"] = homes["longitude"]
    # Months since January 2021, so the model can follow prices over time.
    X["months"] = (homes["date_listed"] - pd.Timestamp("2021-01-01")).dt.days / 30.44
    # One 0/1 column per property type.
    for kind in PROPERTY_TYPES:
        X["is_" + kind] = (homes["house_type_name"] == kind).astype(int)
    return X


=== LIST-AWARE FEATURES ===
def list_aware_features(homes):
    """Features for homes that have a list price: everything above, plus the list price."""
    X = off_market_features(homes)
    X["list_price"] = homes["list_price"]
    return X



In [32]:
from itertools import product
import pandas as pd

round5_configs = [
    {
        "learning_rate": 0.1,
        "max_iter": 400,
        "max_leaf_nodes": leaves,
        "min_samples_leaf": min_leaf,
        "l2_regularization": 0.0
    }
    for leaves, min_leaf in product(
        [31, 45, 63],
        [20, 40]
    )
]

display(pd.DataFrame(round5_configs))

,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization
0,0.1,400,31,20,0.0
1,0.1,400,31,40,0.0
2,0.1,400,45,20,0.0
3,0.1,400,45,40,0.0
4,0.1,400,63,20,0.0
5,0.1,400,63,40,0.0


In [33]:
round5_results = []

for track in ["off_market", "list_aware"]:

    for i, params in enumerate(round5_configs):

        print(
            f"Testing {track}, configuration {i+1}/6",
            flush=True
        )

        result = walk_forward_evaluate(
            params=params,
            track=track,
            listings=listings,
            months=tuning_months
        )

        round5_results.append({
            "Track": track,
            **params,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

round5_results = pd.DataFrame(round5_results)

display(
    round5_results
    .sort_values(["Track", "MdAPE"])
    .round(3)
)

Testing off_market, configuration 1/6
Testing off_market, configuration 2/6
Testing off_market, configuration 3/6
Testing off_market, configuration 4/6
Testing off_market, configuration 5/6
Testing off_market, configuration 6/6
Testing list_aware, configuration 1/6
Testing list_aware, configuration 2/6
Testing list_aware, configuration 3/6
Testing list_aware, configuration 4/6
Testing list_aware, configuration 5/6
Testing list_aware, configuration 6/6


,Track,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MdAPE,MAPE
8,list_aware,0.1,400,45,20,0.0,4.412,5.584
9,list_aware,0.1,400,45,40,0.0,4.429,5.564
10,list_aware,0.1,400,63,20,0.0,4.434,5.593
7,list_aware,0.1,400,31,40,0.0,4.434,5.589
11,list_aware,0.1,400,63,40,0.0,4.448,5.593
6,list_aware,0.1,400,31,20,0.0,4.500,5.626
4,off_market,0.1,400,63,20,0.0,12.542,16.980
5,off_market,0.1,400,63,40,0.0,12.681,17.034
3,off_market,0.1,400,45,40,0.0,12.960,17.206
2,off_market,0.1,400,45,20,0.0,13.116,17.249


In [35]:
from pathlib import Path

save_folder = Path(
    "/content/drive/MyDrive/SSU-Validation/reports"
)

save_folder.mkdir(parents=True, exist_ok=True)

round5_results.to_csv(
    save_folder / "member3_round5_results.csv",
    index=False
)

print("Round 5 saved!")

Round 5 saved!


In [36]:
round6_configs = {
    "off_market": [
        {
            "learning_rate": lr,
            "max_iter": n,
            "max_leaf_nodes": 63,
            "min_samples_leaf": 20,
            "l2_regularization": 0.0
        }
        for lr, n in product([0.05, 0.1], [400, 600])
    ],
    "list_aware": [
        {
            "learning_rate": lr,
            "max_iter": n,
            "max_leaf_nodes": 45,
            "min_samples_leaf": 20,
            "l2_regularization": 0.0
        }
        for lr, n in product([0.05, 0.1], [400, 600])
    ]
}

for track, configs in round6_configs.items():
    print(track)
    display(pd.DataFrame(configs))

off_market


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization
0,0.05,400,63,20,0.0
1,0.05,600,63,20,0.0
2,0.10,400,63,20,0.0
3,0.10,600,63,20,0.0


list_aware


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization
0,0.05,400,45,20,0.0
1,0.05,600,45,20,0.0
2,0.10,400,45,20,0.0
3,0.10,600,45,20,0.0


In [37]:
from pathlib import Path

save_path = Path(
    "/content/drive/MyDrive/SSU-Validation/reports/member3_round6.csv"
)
save_path.parent.mkdir(parents=True, exist_ok=True)

if save_path.exists():
    round6_records = pd.read_csv(save_path).to_dict("records")
else:
    round6_records = []

for track, configs in round6_configs.items():
    for params in configs:

        completed = any(
            r["Track"] == track
            and float(r["learning_rate"]) == params["learning_rate"]
            and int(r["max_iter"]) == params["max_iter"]
            and int(r["max_leaf_nodes"]) == params["max_leaf_nodes"]
            for r in round6_records
        )

        if completed:
            continue

        print(f"Testing {track}: {params}", flush=True)

        result = walk_forward_evaluate(
            params=params,
            track=track,
            listings=listings,
            months=tuning_months
        )

        round6_records.append({
            "Track": track,
            **params,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

        pd.DataFrame(round6_records).to_csv(
            save_path,
            index=False
        )

round6_results = pd.DataFrame(round6_records)

display(
    round6_results.sort_values(["Track", "MdAPE"]).round(3)
)

Testing off_market: {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market: {'learning_rate': 0.05, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market: {'learning_rate': 0.1, 'max_iter': 400, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market: {'learning_rate': 0.1, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing list_aware: {'learning_rate': 0.05, 'max_iter': 400, 'max_leaf_nodes': 45, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing list_aware: {'learning_rate': 0.05, 'max_iter': 600, 'max_leaf_nodes': 45, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing list_aware: {'learning_rate': 0.1, 'max_iter': 400, 'max_leaf_nodes': 45, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing list_aware: {'learning_rate': 0.1, 'max_iter': 600, 'max_leaf_no

,Track,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MdAPE,MAPE
7,list_aware,0.10,600,45,20,0.0,4.398,5.587
6,list_aware,0.10,400,45,20,0.0,4.412,5.584
4,list_aware,0.05,400,45,20,0.0,4.467,5.610
5,list_aware,0.05,600,45,20,0.0,4.467,5.597
3,off_market,0.10,600,63,20,0.0,12.435,16.750
2,off_market,0.10,400,63,20,0.0,12.542,16.980
1,off_market,0.05,600,63,20,0.0,13.074,17.420
0,off_market,0.05,400,63,20,0.0,13.494,17.923


In [38]:
import pandas as pd
from pathlib import Path

print("Data ready:", "listings" in globals())
print("Function ready:", callable(globals().get("walk_forward_evaluate")))
print("Months ready:", "tuning_months" in globals())
print("Drive ready:", Path("/content/drive/MyDrive").exists())

Data ready: True
Function ready: True
Months ready: True
Drive ready: True


In [39]:
from itertools import product

round7_configs = {

    "off_market": [
        {
            "learning_rate": 0.1,
            "max_iter": n,
            "max_leaf_nodes": leaves,
            "min_samples_leaf": 20,
            "l2_regularization": 0.0
        }
        for n, leaves in product(
            [500, 600, 700],
            [63, 80]
        )
    ],

    "list_aware": [
        {
            "learning_rate": 0.1,
            "max_iter": n,
            "max_leaf_nodes": 45,
            "min_samples_leaf": 20,
            "l2_regularization": l2
        }
        for n, l2 in product(
            [400, 500, 600],
            [0.0, 1.0]
        )
    ]
}

for track, configs in round7_configs.items():
    print(f"\n{track}: {len(configs)} configurations")
    display(pd.DataFrame(configs))


off_market: 6 configurations


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization
0,0.1,500,63,20,0.0
1,0.1,500,80,20,0.0
2,0.1,600,63,20,0.0
3,0.1,600,80,20,0.0
4,0.1,700,63,20,0.0
5,0.1,700,80,20,0.0



list_aware: 6 configurations


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization
0,0.1,400,45,20,0.0
1,0.1,400,45,20,1.0
2,0.1,500,45,20,0.0
3,0.1,500,45,20,1.0
4,0.1,600,45,20,0.0
5,0.1,600,45,20,1.0


In [40]:
from pathlib import Path

save_path = Path(
    "/content/drive/MyDrive/SSU-Validation/reports/member3_round7.csv"
)

save_path.parent.mkdir(parents=True, exist_ok=True)

# If interrupted, resume saved progress
if save_path.exists():
    round7_records = pd.read_csv(save_path).to_dict("records")
else:
    round7_records = []

for track, configs in round7_configs.items():

    for i, params in enumerate(configs):

        completed = any(
            r["Track"] == track
            and all(
                abs(float(r[key]) - float(value)) < 1e-9
                for key, value in params.items()
            )
            for r in round7_records
        )

        if completed:
            print(f"Skipping completed: {track}, {i+1}/6")
            continue

        print(
            f"Testing {track}, configuration {i+1}/6: {params}",
            flush=True
        )

        result = walk_forward_evaluate(
            params=params,
            track=track,
            listings=listings,
            months=tuning_months
        )

        round7_records.append({
            "Track": track,
            **params,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

        # Save after each configuration
        pd.DataFrame(round7_records).to_csv(
            save_path,
            index=False
        )

round7_results = pd.DataFrame(round7_records)

display(
    round7_results
    .sort_values(["Track", "MdAPE"])
    .round(3)
)

print("Round 7 completed and saved!")

Testing off_market, configuration 1/6: {'learning_rate': 0.1, 'max_iter': 500, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market, configuration 2/6: {'learning_rate': 0.1, 'max_iter': 500, 'max_leaf_nodes': 80, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market, configuration 3/6: {'learning_rate': 0.1, 'max_iter': 600, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market, configuration 4/6: {'learning_rate': 0.1, 'max_iter': 600, 'max_leaf_nodes': 80, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market, configuration 5/6: {'learning_rate': 0.1, 'max_iter': 700, 'max_leaf_nodes': 63, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing off_market, configuration 6/6: {'learning_rate': 0.1, 'max_iter': 700, 'max_leaf_nodes': 80, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
Testing list_aware, configuration 1/6: {'learning_rate': 0.1, 'max_iter': 400, 'max_leaf_nodes

,Track,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MdAPE,MAPE
8,list_aware,0.1,500,45,20,0.0,4.398,5.587
10,list_aware,0.1,600,45,20,0.0,4.398,5.587
9,list_aware,0.1,500,45,20,1.0,4.404,5.605
11,list_aware,0.1,600,45,20,1.0,4.404,5.605
7,list_aware,0.1,400,45,20,1.0,4.404,5.605
6,list_aware,0.1,400,45,20,0.0,4.412,5.584
3,off_market,0.1,600,80,20,0.0,12.249,16.819
5,off_market,0.1,700,80,20,0.0,12.249,16.819
1,off_market,0.1,500,80,20,0.0,12.291,16.853
4,off_market,0.1,700,63,20,0.0,12.298,16.671


Round 7 completed and saved!


In [41]:
best_round7 = (
    round7_results
    .sort_values("MdAPE")
    .groupby("Track")
    .head(1)
    .reset_index(drop=True)
)

display(best_round7.round(3))

,Track,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MdAPE,MAPE
0,list_aware,0.1,600,45,20,0.0,4.398,5.587
1,off_market,0.1,600,80,20,0.0,12.249,16.819


In [42]:
previous_best = {
    "off_market": 12.435,
    "list_aware": 4.398
}

best_round7["Round 6 MdAPE"] = (
    best_round7["Track"].map(previous_best)
)

best_round7["Improvement (pp)"] = (
    best_round7["Round 6 MdAPE"]
    - best_round7["MdAPE"]
)

display(best_round7.round(3))

,Track,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MdAPE,MAPE,Round 6 MdAPE,Improvement (pp)
0,list_aware,0.1,600,45,20,0.0,4.398,5.587,4.398,-0.000
1,off_market,0.1,600,80,20,0.0,12.249,16.819,12.435,0.186


In [43]:
best_round7.to_csv(
    "/content/drive/MyDrive/SSU-Validation/reports/member3_round7_best.csv",
    index=False
)

print("Best Round 7 configurations saved!")

Best Round 7 configurations saved!


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

train = listings[
    listings["date_sold"] < pd.Timestamp("2025-09-01")
].copy()

results_iter = []

for track, leaves in [
    ("off_market", 80),
    ("list_aware", 45)
]:
    feature_fn = (
        features.off_market_features
        if track == "off_market"
        else features.list_aware_features
    )

    X = feature_fn(train)
    y = np.log(train["sold_price"])

    for max_iter in [400, 500, 600, 700]:

        model = HistGradientBoostingRegressor(
            learning_rate=0.1,
            max_iter=max_iter,
            max_leaf_nodes=leaves,
            min_samples_leaf=20,
            l2_regularization=0,
            random_state=0
        )

        model.fit(X, y)

        results_iter.append({
            "Track": track,
            "Requested Iterations": max_iter,
            "Actual Iterations": model.n_iter_,
            "Early Stopping": model.do_early_stopping_
        })

display(pd.DataFrame(results_iter))

NameError: name 'listings' is not defined

In [2]:
from google.colab import drive
from pathlib import Path
import shutil

# Connect Google Drive
drive.mount("/content/drive")

# Restore project from Google Drive
source = Path("/content/drive/MyDrive/SSU-Validation")
destination = Path("/content/SSU-Validation")

if not source.exists():
    raise FileNotFoundError("Project backup not found in Google Drive")

shutil.copytree(
    source,
    destination,
    dirs_exist_ok=True
)

%cd /content/SSU-Validation

print("Project restored!")

Mounted at /content/drive
/content/SSU-Validation
Project restored!


In [3]:
import numpy as np
import pandas as pd
import data
import features

from sklearn.ensemble import HistGradientBoostingRegressor

listings = data.load("listings")

print("Data loaded successfully!")
print("Number of rows:", len(listings))

listings.parquet: reconstructing file:   0%|          |  0.00B / 32.9MB            

listings.parquet: downloading bytes:           |  0.00B            

Data loaded successfully!
Number of rows: 150000


In [4]:
import numpy as np
import pandas as pd
import features
from sklearn.ensemble import HistGradientBoostingRegressor

train = listings[
    listings["date_sold"] < pd.Timestamp("2025-09-01")
].copy()

results_iter = []

for track, leaves in [
    ("off_market", 80),
    ("list_aware", 45)
]:

    feature_fn = (
        features.off_market_features
        if track == "off_market"
        else features.list_aware_features
    )

    X = feature_fn(train)
    y = np.log(train["sold_price"])

    for max_iter in [400, 500, 600, 700]:

        model = HistGradientBoostingRegressor(
            learning_rate=0.1,
            max_iter=max_iter,
            max_leaf_nodes=leaves,
            min_samples_leaf=20,
            l2_regularization=0,
            random_state=0
        )

        model.fit(X, y)

        results_iter.append({
            "Track": track,
            "Requested Iterations": max_iter,
            "Actual Iterations": model.n_iter_,
            "Early Stopping": model.do_early_stopping_
        })

display(pd.DataFrame(results_iter))

,Track,Requested Iterations,Actual Iterations,Early Stopping
0,off_market,400,400,True
1,off_market,500,500,True
2,off_market,600,591,True
3,off_market,700,591,True
4,list_aware,400,400,True
5,list_aware,500,465,True
6,list_aware,600,465,True
7,list_aware,700,465,True


In [5]:
save_path = (
    "/content/drive/MyDrive/SSU-Validation/"
    "reports/member3_early_stopping_check.csv"
)

pd.DataFrame(results_iter).to_csv(save_path, index=False)

print("Early stopping analysis saved!")

Early stopping analysis saved!


In [6]:
round8_configs = {
    "off_market": [
        {
            "learning_rate": 0.1,
            "max_iter": 600,
            "max_leaf_nodes": 80,
            "min_samples_leaf": 20,
            "l2_regularization": 0.0,
            "early_stopping": es
        }
        for es in [True, False]
    ],

    "list_aware": [
        {
            "learning_rate": 0.1,
            "max_iter": 500,
            "max_leaf_nodes": 45,
            "min_samples_leaf": 20,
            "l2_regularization": 0.0,
            "early_stopping": es
        }
        for es in [True, False]
    ]
}

for track, configs in round8_configs.items():
    print(track)
    display(pd.DataFrame(configs))

off_market


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,early_stopping
0,0.1,600,80,20,0.0,True
1,0.1,600,80,20,0.0,False


list_aware


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,early_stopping
0,0.1,500,45,20,0.0,True
1,0.1,500,45,20,0.0,False


In [12]:
round8_results = []

for track, configs in round8_configs.items():

    for params in configs:

        print(
            f"Testing {track}, "
            f"early_stopping={params['early_stopping']}",
            flush=True
        )

        result = walk_forward_evaluate(
            params=params,
            track=track,
            listings=listings,
            months=tuning_months
        )

        round8_results.append({
            "Track": track,
            **params,
            "MdAPE": result["MdAPE"],
            "MAPE": result["MAPE"]
        })

round8_results = pd.DataFrame(round8_results)

display(
    round8_results
    .sort_values(["Track", "MdAPE"])
    .round(3)
)

Testing off_market, early_stopping=True
Testing off_market, early_stopping=False
Testing list_aware, early_stopping=True
Testing list_aware, early_stopping=False


,Track,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,early_stopping,MdAPE,MAPE
2,list_aware,0.1,500,45,20,0.0,True,4.398,5.587
3,list_aware,0.1,500,45,20,0.0,False,4.432,5.609
1,off_market,0.1,600,80,20,0.0,False,12.139,16.590
0,off_market,0.1,600,80,20,0.0,True,12.249,16.819


In [8]:
def walk_forward_evaluate(params, track, listings, months):
    all_errors = []
    monthly_results = []

    if track == "off_market":
        feature_function = features.off_market_features
    else:
        feature_function = features.list_aware_features

    for month in months:
        train = listings[
            listings["date_sold"] < month.start_time
        ]

        validation = listings[
            listings["date_sold"].dt.to_period("M") == month
        ]

        model = HistGradientBoostingRegressor(
            random_state=0,
            **params
        )

        X_train = feature_function(train)
        y_train = np.log(train["sold_price"])

        model.fit(X_train, y_train)

        hidden = data.hide_answers(validation, track)
        X_val = feature_function(hidden)

        predictions = np.exp(model.predict(X_val))
        actual = validation["sold_price"].to_numpy()

        ape = np.abs(predictions - actual) / actual * 100

        all_errors.extend(ape)

        monthly_results.append({
            "Month": str(month),
            "MdAPE": np.median(ape),
            "MAPE": np.mean(ape),
            "Homes": len(validation)
        })

    return {
        "MdAPE": np.median(all_errors),
        "MAPE": np.mean(all_errors),
        "Monthly": pd.DataFrame(monthly_results)
    }

print("Walk-forward function restored!")

Walk-forward function restored!


In [9]:
print(
    "Function ready:",
    callable(globals().get("walk_forward_evaluate"))
)

print("Data ready:", "listings" in globals())
print("Months ready:", "tuning_months" in globals())

Function ready: True
Data ready: True
Months ready: False


In [10]:
import pandas as pd

tuning_months = pd.period_range(
    "2025-09",
    "2025-11",
    freq="M"
)

print("Tuning months restored!")
print(tuning_months)

Tuning months restored!
PeriodIndex(['2025-09', '2025-10', '2025-11'], dtype='period[M]')


In [11]:
print("Function ready:", callable(globals().get("walk_forward_evaluate")))
print("Data ready:", "listings" in globals())
print("Months ready:", "tuning_months" in globals())
print("Configs ready:", "round8_configs" in globals())

Function ready: True
Data ready: True
Months ready: True
Configs ready: True


In [13]:
from pathlib import Path

save_folder = Path(
    "/content/drive/MyDrive/SSU-Validation/reports"
)

save_folder.mkdir(parents=True, exist_ok=True)

round8_results.to_csv(
    save_folder / "member3_round8_results.csv",
    index=False
)

print("Round 8 saved successfully!")

Round 8 saved successfully!
